# AI: Delegate the full problem {#sec-delegate-the-full-problem}

![](images/under-construction.png){width=20%}

By now I am sure you appreciate the usefulness of tests. You have been using the tests I wrote for the programming projects, but you have also written a few of your own. 



Your code is wrong if it does not pass all the tests. 



You code is correct if it passes all tests.



Tests made correctly Because tests fail if the code does not passA set of testYou can think of tests as *testing*




# AI: Tests are the contract {#sec-tests-are-the-contract}

You already know how to write a test: a claim about a function that the machine can check, like `assert gc_content('GGCC') == 1.0`. This week that skill becomes your most important tool for working with an assistant. A test is a specification you can execute, and a specification is a contract. Seen that way, getting code out of a machine you cannot fully trust becomes manageable, because the test decides whether the code is right. The assistant does not decide it, and neither do you.

Before you can judge whether code is correct, you must decide what correct means, and that decision comes before any code. A request for a function that finds genes can be neither right nor wrong, because it does not say what output to expect for what input. A specification says exactly that. Written as a comment or a docstring, it is a promise the code can quietly break, as you saw last week. Written as a test, such as `assert reverse_complement('ATGC') == 'GCAT'`, it is a promise the machine enforces: the moment the code breaks it, you are told.

So the discipline is about order. The natural order is to ask for the code, glance at it and hope. The disciplined order is the reverse: choose concrete examples, write them as tests, then ask for the code, then run the tests. Writing the tests first matters more than it seems. If you write them after seeing a plausible function, you will, without meaning to, write tests it happens to pass, and end up checking that the code does what the code does. Writing them first keeps the decision about what correct means with you, before the assistant has a chance to bias it.

You turn a vague request into a specification by choosing examples and working out their answers by hand. For the GC content of a strand, you write down that `gc_content('GGCC')` should be 1.0, `gc_content('ATAT')` 0.0 and `gc_content('GGAT')` 0.5 before you prompt for anything. Each example becomes one line of your test, and the contract is only as strong as its examples. Gentle examples let a wrong function pass, which is last week's happy path trap again. A strong contract includes awkward inputs on purpose: the empty string, a single base, lowercase letters, a character that should not appear.

Writing tests first also forces decisions you would otherwise leave to the assistant to guess. Those decisions live in the awkward cases, and a fluent assistant settles them silently, in whatever way its training leans. Should `is_gc_rich('GGAT')`, with a GC content of exactly 0.5, return `True` or `False`? There is no universal answer. It is your decision, and writing the test first makes you take it deliberately. If you do not, the assistant picks `>` or `>=` for you, and you cannot tell whether its choice matches your intention, because you never formed one. The same happens when a function returns the most common base and two bases are tied, as `G` and `C` are in `GC`:

```python
def most_common_base(dna):
    return max(set(dna), key=dna.count)

print(most_common_base('GC'))
```

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Before you run it, write down which base you expect to come back, and why. Then run it. The real question is not about the answer: could you have predicted it from the code alone, without knowing how `set` orders its elements and how `max` breaks a tie? If not, the behavior on a tie is something that fell out of the code, and nobody decided it. Write the single `assert` that would take the decision back, and notice that writing it forces you to make the decision.

The base you get back is an accident of how `max` breaks a tie over a set whose order you should not rely on. Which base it is does not matter. What matters is that the tie is a decision about what the function means, that the assistant will make it for you without saying so, and that the only way to take it back is a test written before any code exists. Writing tests first is mostly about doing the thinking in advance, for the cases that need it.

This is why a test is better called a contract than a check. A contract says: I do not care how you do the job, but the result must meet these conditions. That is the relationship you want with an assistant you can neither fully trust nor inspect. You write the contract in a form the machine enforces, the assistant supplies an implementation, and the contract decides whether it is acceptable. The implementation also becomes replaceable. You can throw away the assistant's version, write your own or accept a faster one later, and the same tests keep guarding you. The tests outlive any one attempt at the code.

You have met this idea before, without writing the contracts yourself. Every tick and cross from the check widget on a project was a clause of a contract enforced against your code. From now on you meet tests in three ways: through the widget on the projects, as `assert` statements you write to pin down functions, and through the `pytest` command in the terminal, whose report lists which contracts held and which broke, so you can read the verdict without the widget.

Here is the whole discipline on a small problem: a contract for `gc_content`, written before any implementation exists, with the awkward case decided on purpose.

```python
assert gc_content('GGCC') == 1.0
assert gc_content('ATAT') == 0.0
assert gc_content('GGAT') == 0.5
assert gc_content('ggcc') == 1.0     # lowercase counts too: a deliberate decision
```

Only now do you ask the assistant for the function. A version that counts only the capital letters `G` and `C` passes the first three clauses and fails the fourth. That failure is the contract doing its job. It catches a decision the assistant made silently that differs from the one you made deliberately, and without the fourth clause you would never have known you disagreed.

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Turn a vague request into a contract. Pick a small function you might ask an assistant for, such as one that counts how often a given codon occurs in a longer strand. Before any code exists, write three examples with the answers worked out by hand, and turn each into an `assert`. Make at least one of them awkward, such as an overlapping occurrence or a codon that does not occur at all, and be ready to say what decision it forces. Keep this contract; you will use it again below.

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

This one is without an assistant, so that the mechanics are yours. Take a function you wrote earlier in the course and write two tests for it: one you are confident it passes, and one you expect it to fail, perhaps on a case it never handled. Run both, and feel the difference between a test that confirms and a test that accuses. A contract that only ever confirms protects you from nothing.

#### Exercise

[AI: Drafter](../intro/course-introduction.qmd#sec-badge-drafter){.small}

Now run the full discipline on the contract you wrote two exercises ago. Ask the assistant for the function, giving it the purpose and the signature but not your tests. Run your tests against what it produces. If they all pass, you have earned a belief in the code, limited to the cases you specified. If one fails, you found the disagreement before the code was ever used, which is the reason for writing the contract first.

#### Exercise

[AI: Drafter](../intro/course-introduction.qmd#sec-badge-drafter){.small}

Now aim a contract at a decision instead of a bug. Write a contract for `is_gc_rich(dna)` that settles two things the request leaves open: a strand that is exactly half GC, like `'GGAT'`, and the empty strand. Decide both and write an `assert` for each. Then ask the assistant for the function using only a prose description that says nothing about either case, and run your contract. Whatever it chose, it chose silently, and you can tell whether you agree only because you decided first. Record which of your two decisions, if either, it got wrong.

#### Exercise

[AI: Drafter](../intro/course-introduction.qmd#sec-badge-drafter){.small}

Last, watch the contract outlive the code. Ask the assistant for a second, deliberately different implementation of the same function, for example one without a loop, or one a beginner would find easier to read, and run your unchanged tests against it. Two pieces of code, one contract, the same verdict: the implementation is replaceable and the specification is not, which is why the tests are worth more than any particular function you get out of a machine.

For your logbook this week, record one specification you wrote as tests before requesting any code, the awkward case you had to decide, and whether the assistant's code agreed with your decision or quietly disagreed. If it agreed on everything, add the awkward case you wish you had thought to specify.



You now hold three skills. You can specify a function as tests before any code exists. You can judge a function the assistant produced against a short list of questions. And you can break a larger task into named, testable pieces and write that plan down before prompting. In the finale these become one way of working, applied to a task deliberately larger than anything you could write on your own. That is the purpose of the exercise. The role you step into this week is the one the course has been climbing toward: delegating a real job to the assistant while remaining fully responsible for the result.

Beyond your unaided reach does not mean that you must suddenly become a better programmer. You are not expected to write the finale from a blank page. You are expected to produce it, correct and understood, by driving the assistant through a plan you designed and verifying every piece it hands back. A professional programmer would write it unaided. You will decompose it, specify it, delegate the typing and check everything. Done well, the result is just as correct, and you understand it just as well, even though you could not have typed it yourself. Closing that gap is what this course is for.

The method is the full loop you already have, run once for each piece of the plan. Before prompting, decompose the task into named functions with stated inputs and outputs. Then take the pieces in dependency order. For each one, write its contract as a few tests, seeded from the example in your plan and strengthened with awkward cases. Prompt the assistant for that one function, giving it the purpose, the inputs and outputs, and the tests it must satisfy. Read what it returns against your judging questions, and run the tests. If they pass, the piece is done, and you build the next one on top of it. If they fail, send a specific change request naming the failing input, the wrong output, the right output and, as far as you can see it, the reason, and run the tests again. Nothing in this loop is new. What is new is running it many times, on a task you could not have written, and watching a program you understand assemble from pieces you have each checked.

When a piece fails and you send a change request, the loop should converge within a round or two. If the assistant keeps failing in the same way, or fixes your case while breaking another, do not keep pushing the same prompt. It usually means one of two things. Either your specification is wrong or incomplete, and you should fix the tests and leave the code alone. Or you now understand the piece well enough to write it faster than you can delegate it. Knowing when to take the keyboard back is part of the skill. The assistant is there to save you effort, and arguing with it about a piece you could finish in five minutes saves you none.

Two things about verification change scale in the finale. First, the project comes with a test suite, and it is tempting to treat passing it as the finish line. The suite is a floor. It holds the cases its author knew to check, and the assistant is good at satisfying visible tests while failing invisible ones. Add your own tests, especially for the nasty cases, and treat the provided suite as the start of your checking. Second, passing tests, even your own, are necessary but not sufficient. They show that the code works on the cases you thought of and say nothing about the others. The backstop is your own understanding of each piece, which is why you never accept code you cannot read. A function that passes every test but that you cannot explain is not finished, because on an input no test covered, your understanding is all that stands between a wrong answer and a published result.

This is a matter of responsibility, the part of delegation that does not transfer. You can hand the assistant the typing, but not the ownership of the result. When you submit the finale, you must be able to stand behind every function in it and explain what it does and why it is correct. If someone asks how a function works, "the AI wrote it" is no answer: it admits that you delegated the thinking along with the typing. The specifying, judging and planning exist so that you can use the assistant heavily and still own the outcome. Delegating well means exercising responsibility through verification instead of through typing.

To make your verification visible, keep a prompt journal while you build the finale. The weekly logbook is a broad reflection. The prompt journal is a narrow running record, piece by piece, of what you asked for, where the answer was wrong, how you caught it and what you did about it. It makes you notice failures you might otherwise skim past, and by the end it documents your judgment at work, which is what the finale is meant to demonstrate. In this course, a finale that runs is worth less than one that runs and comes with a frank account of the places the assistant got it wrong and how you knew.

You will do most of this on the real project, so the exercises here are small and meant to sharpen particular moves in the loop.

#### Exercise

[AI: Developer](../intro/course-introduction.qmd#sec-badge-developer){.small}

Run one full cycle on one piece, in the safe order. Before prompting, write the acceptance criteria for one function from your plan: what it must return for which inputs, awkward cases included, in a form the machine can check. Then prompt the assistant for the function, giving it the purpose, the signature and nothing you have not decided. Read what comes back, and run the criteria. Notice that you wrote the criteria with no code in front of you to anchor your idea of what correct means. That is what separates delegating a piece from hoping.

#### Exercise

[AI: Developer](../intro/course-introduction.qmd#sec-badge-developer){.small}

Prompt the assistant for one function from your plan. When its first attempt fails one of your tests, write the change request you would send back: the failing input, the wrong output, the correct output and the reason. Notice how much of it you already knew from writing the test first. Send it. If the second version passes, the loop converged. If the third does not, stop and go to the next exercise.

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Take the keyboard back. Find a piece where the loop did not converge in two rounds, or where you can already see the answer, and write it yourself. Time both approaches if you can. This trains the judgment about when delegating has stopped paying, which is a skill just as specifying and judging are. Twenty minutes arguing with an assistant over a function you could type in five saves you nothing, and noticing that quickly is part of using it well.

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Take a function that has just passed the provided tests, write one more test of your own for a nasty case the suite did not include, and run it. If it passes, you have strengthened the contract. If it fails, you have caught a bug that would otherwise have shipped, and seen for yourself that the provided suite is a floor.

#### Exercise

[AI: Developer](../intro/course-introduction.qmd#sec-badge-developer){.small}

Finally, the exercise that decides whether you keep the badge. Pick the function in your finale that you delegated most heavily, the one you are least sure you could have written yourself. With the assistant closed, write one paragraph in your own words explaining how it works, line by line. Its tests already tell you what it does. If you cannot explain how it does it, the function is not finished, whatever its tests say, because you cannot own a result you cannot explain. Read it until you can, and write the paragraph again. Delegation transfers the typing and keeps the responsibility with you.

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Write one prompt journal entry for a piece you built, recording where the assistant was wrong and how the machine, or your reading, revealed it.

For your logbook this week, describe what it was like to produce something you could not have written unaided. Which part of the loop did the most work in making the task possible: the planning, the specifying, the judging or the iterating? Name one function in your finale that you could now explain in full to someone else, and one you are not yet sure you could. The second kind is where your remaining work is.